# HW6 — Final-Project Proposal & Baseline Checkpoint

**FIN 7057 · Due Weeks 10-11 · 100 pts**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/homework/hw6/hw6_starter.ipynb)

This is a project checkpoint, not the final project. Define the question, demonstrate point-in-time
data handling, and produce one development-validation baseline. Reserve a final lockbox and do not
open it in this assignment. Read the [complete task and rubric](https://github.com/smc77/uc_finmlai/blob/main/homework/hw6/README.md) before beginning.


## Part A — Research question, track & prior evidence

**Research question (TODO):** _…_

**Track (TODO):** Markets / asset management or corporate / credit / text. Explain why
the target, data, metric, and decision use belong together.

**Provisional contribution (TODO):** Is this primarily a replication, extension, comparison,
audit, or applied evaluation? State the narrow contribution without claiming novelty you have
not established.

**Literature map (TODO):** Add at least four credible sources. Verify each citation yourself.

| Source | Question and evidence | Narrow finding | Relevance to this project |
|---|---|---|---|
| _Author (year)_ | _…_ | _…_ | _…_ |

**Report pathway and outline (TODO):** Choose the integrated Jupyter report or analysis plus a
separate paper. List the planned paper sections. See [Report pathways](https://github.com/smc77/uc_finmlai/blob/main/project/REPORT_PATHWAYS.md).


## Part B — Data pipeline & point-in-time availability

Document the source, sample, unit of observation, target, and the timestamp at which every input
became usable. A markets project should test feature invariance under future perturbation. A
corporate/credit project should account for filing, restatement, and label dates. A text project
should use publication/availability timestamps and guard against revised or backfilled documents.

Complete and extend this table before coding:

| Input | Source and version | Unit of observation | Subject time | Public time | Feature-ready time | Revision or leakage control |
|---|---|---|---|---|---|---|
| _TODO_ | _TODO_ | _TODO_ | _TODO_ | _TODO_ | _TODO_ | _TODO_ |


## Part C — Features + a working baseline (the coding checkpoint)

Replace or adapt the scaffold below for your project. Before the result, state your target, horizon,
unit of observation, training and validation boundaries, final lockbox boundary, and the availability
check appropriate to your track. The included markets scaffold is an example interface, not a finished
submission. It attempts a real Yahoo Finance download; a synthetic fallback keeps the cell inspectable
offline but **does not satisfy the assignment and will fail the final self-check**. If needed in Colab,
install `yfinance` in a new cell with `%pip install -q yfinance`, then restart and run all.


In [ ]:
# Runnable markets scaffold. Corporate/credit/text projects may replace this cell, but must
# preserve the track-neutral interface at the bottom: TRACK, METRIC_NAME, baseline_score,
# REAL_DATA_USED, availability_checked, training_end, validation_start, development_end, LOCKBOX_START,
# and LOCKBOX_OPENED.
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import make_pipeline

SEED=42; np.random.seed(SEED)
TRACK='markets'  # TODO: 'markets' or 'corporate_credit_text'
LOCKBOX_START=pd.Timestamp('2023-01-01')
LOCKBOX_OPENED=False

DATA_SOURCE=None

def load_prices(t='SPY',s='2012-01-01',e='2025-01-01'):
    global DATA_SOURCE
    try:
        import yfinance as yf
        df=yf.download(t,start=s,end=e,progress=False,auto_adjust=True)
        if len(df):
            c=df['Close'].squeeze(); c.name='close'; c.index.name='date'
            DATA_SOURCE=f'Yahoo Finance: {t}, downloaded for {s} through {e}'
            print(f'loaded {len(c)} real rows from {DATA_SOURCE}')
            return c.to_frame()
        raise RuntimeError('empty')
    except Exception as ex:
        DATA_SOURCE=f'synthetic fallback ({type(ex).__name__})'
        print('WARNING:',DATA_SOURCE,'— replace this with real project data before submission')
        idx=pd.bdate_range(s,e,inclusive='left')
        rng=np.random.default_rng(SEED)
        return pd.DataFrame({'close':100*np.exp(np.cumsum(rng.standard_normal(len(idx))*.01))},
                            index=idx)

def build_features(prices):
    p=prices.copy(); p['ret']=np.log(p['close']).diff()
    f=pd.DataFrame(index=p.index)
    for w in (5,21,63): f[f'mom{w}']=p['ret'].rolling(w).mean().shift(1)
    f['vol21']=p['ret'].rolling(21).std().shift(1)
    return f

def assert_no_lookahead(make_features,prices,t0=.6):
    base=make_features(prices); pert=prices.copy(); t=int(len(prices)*t0)
    pert.iloc[t+1:,pert.columns.get_loc('close')]*=1.25
    d=(base.iloc[:t+1]-make_features(pert).iloc[:t+1]).abs().to_numpy()
    d=d[np.isfinite(d)]
    return float(d.max()) if d.size else 0.

px=load_prices(); REAL_DATA_USED=not DATA_SOURCE.startswith('synthetic fallback')
px['ret']=np.log(px['close']).diff()
feat=build_features(px); target=px['ret'].shift(-1).rename('y')
data=feat.join(target).dropna(); cols=list(feat.columns)
development=data[data.index<LOCKBOX_START]
lockbox=data[data.index>=LOCKBOX_START]  # reserve only; do not score it here
cut=int(len(development)*.7)
tr,va=development.iloc[:cut],development.iloc[cut:]

model=GridSearchCV(
    make_pipeline(StandardScaler(),Ridge()),
    {'ridge__alpha':np.logspace(-3,3,13)},cv=TimeSeriesSplit(5),
    scoring='neg_mean_squared_error').fit(tr[cols],tr['y'])
validation_pred=model.predict(va[cols])

# Track-neutral interface used by the self-check.
METRIC_NAME='single-series Spearman correlation'
baseline_score=float(spearmanr(validation_pred,va['y']).statistic)
availability_checked=assert_no_lookahead(build_features,px)<1e-8
training_end=tr.index.max()
development_end=development.index.max()
validation_start=va.index.min()
print(f'development validation {METRIC_NAME}: {baseline_score:+.4f}')
print(f'lockbox reserved from {LOCKBOX_START.date()} ({len(lockbox)} rows; not scored)')


## Part D — Evaluation plan

TODO: Name the primary metric appropriate to your track, the baseline it must beat, the
development-validation design, uncertainty estimate, and decision costs or error asymmetries.
State the final lockbox boundary and leave it unopened.


## Part E — Risks & kill criteria

TODO: Explain the project-specific data, modelling, regime, and deployment failures completely. Prespecify concrete
kill criteria before opening the final lockbox. A correct negative result must remain a possible
outcome.


## Extensions (optional)

The required core is everything above, and the self-check tests only that. These extensions are optional and **not** required for full marks — a clean, well-reasoned core beats a sprawling one. If you want to go further:

- Add a second baseline (a different model or feature set) for comparison.
- Test sensitivity to the sample period (e.g., drop or shift the training window).
- Sketch the decision layer your final project will use, and a first metric for it.

## Self-check (auto-graded)

Run this cell **last**, after the whole notebook. Every line must print **PASS** before you submit — it verifies the mechanical requirements (real data, reproducibility, leak-free features, time-ordered splits, metrics computed). Your written answers and judgment are graded by hand.

In [ ]:
# ================= SELF-CHECK (auto-graded) =================
def _check(name,fn,hint=""):
    try:
        ok=bool(fn())
    except Exception as e:
        ok,hint=False,hint or f"({type(e).__name__}: {e})"
    print(("PASS  " if ok else "FAIL  ")+name+("" if ok else "   -> "+hint))
    return ok
_r=[]
_r.append(_check("recognized project track",lambda:TRACK in {'markets','corporate_credit_text'},"set TRACK to one of the two project tracks"))
_r.append(_check("real project data loaded",lambda:REAL_DATA_USED is True,"replace the synthetic fallback and set REAL_DATA_USED only after real data load"))
_r.append(_check("finite development-validation baseline",lambda:np.isfinite(baseline_score) and bool(METRIC_NAME),"set METRIC_NAME and baseline_score"))
_r.append(_check("point-in-time availability check documented in code",lambda:availability_checked is True,"set availability_checked only after the track-appropriate check passes"))
_r.append(_check("validation follows training",lambda:training_end<validation_start,"record strictly ordered training and validation boundaries"))
_r.append(_check("final lockbox follows development",lambda:development_end<LOCKBOX_START,"reserve a later final lockbox"))
_r.append(_check("final lockbox remains unopened",lambda:LOCKBOX_OPENED is False,"do not evaluate the lockbox in HW6"))
print(f"\n{sum(_r)}/{len(_r)} mechanical checks passed.")
assert all(_r),"Fix the FAIL items above before submitting."


## AI-use disclosure & reproducibility

State whether and how AI was used, the data and package environment, fixed seeds, sample boundaries,
and how another reader can rebuild the development-validation result. You are responsible for every
claim and must be able to explain it.
